# Multi-hop Retrieval

**Multi-hop Retrieval**이란?
단일 검색으로 답을 찾을 수 없는 복합적인 질문에 대해, **단계적(Step-by-Step)으로 정보를 검색**하여 최종 답을 도출하는 기법이다.

예를 들어:
- 질문: "iPhone을 만든 회사의 본사가 위치한 도시는 어디인가?"
- 1단계(Hop 1): "iPhone을 만든 회사"를 검색 -> **Apple Inc.** 찾음
- 2단계(Hop 2): "Apple Inc.의 본사가 위치한 도시"를 검색 -> **Cupertino** 찾음
- 최종 답: **Cupertino**

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['COHERE_API_KEY'] = os.getenv('COHERE_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')


In [2]:
import pandas as pd

document_df = pd.read_csv('documents_multihop_v2.csv')
queries_df = pd.read_csv('queries_multihop_v2.csv')
queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,아이폰을 만든 회사의 본사가 위치한 도시는 어디인가요?,D1=1;D2=1;D3=1
1,Q2,아이폰을 설계한 회사의 현재 CEO는 누구인가요?,D1=1;D2=1;D4=1
2,Q3,애플 본사가 위치한 도시는 어느 주에 있나요?,D2=1;D3=1
3,Q4,갤럭시 스마트폰을 만드는 회사의 현재 회장은 누구인가요?,D6=1;D7=1;D8=1
4,Q5,알파고를 개발한 회사의 본사가 위치한 도시는 어디인가요?,D14=1;D15=1
5,Q6,BTS가 소속된 기획사를 설립한 사람은 누구인가요?,D16=1;D17=1
6,Q7,영화 기생충을 연출한 감독이 수상한 영화제는 어디인가요?,D21=1;D22=1;D23=1
7,Q8,손흥민 선수가 소속된 팀의 연고지는 어디인가요?,D26=1;D27=1
8,Q9,토트넘 홋스퍼 FC가 위치한 도시가 수도인 나라는 어디인가요?,D27=1;D28=1
9,Q10,삼성전자를 창립한 사람이 세운 그룹의 이름은 무엇인가요?,D10=1


In [3]:
pd.set_option('display.max_colwidth', None)
document_df

,doc_id,content
0,D1,아이폰(iPhone) 스마트폰 시리즈는 애플(Apple Inc.)에 의해 설계 및 마케팅되었습니다.
1,D2,애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.
2,D3,"쿠퍼티노(Cupertino)는 미국 캘리포니아주에 위치한 도시로, 애플의 본거지로 알려져 있습니다."
3,D4,팀 쿡(Tim Cook)은 2011년 스티브 잡스의 뒤를 이어 애플(Apple Inc.)의 CEO가 되었습니다.
4,D5,스티브 잡스(Steve Jobs)는 애플(Apple Inc.)의 공동 창업자였으며 2011년까지 CEO를 역임했습니다.
5,D6,갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.
6,D7,삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.
7,D8,이재용은 현재 삼성전자의 회장직을 맡고 있습니다.
8,D9,"서울은 대한민국의 수도이자 최대 도시로, 한강이 흐르고 있습니다."
9,D10,삼성전자는 1938년 이병철에 의해 창립된 삼성그룹의 계열사입니다.


### 벡터스토어 생성 및 문서 업로드

In [4]:
from pinecone import Pinecone, ServerlessSpec # Pinecone 클라이언트, 인덱스 스펙설정

pc = Pinecone()
print(pc.list_indexes().names()) # 인덱스 이름

# 인덱스명 ir-multihop
if 'ir-multihop' not in pc.list_indexes().names():
    # 인덱스 생성
    pc.create_index(
        name = 'ir-multihop',
        dimension = 1536,
        metric = 'cosine',       # 유사도 기준
        spec = ServerlessSpec(
            region= 'us-east-1',
            cloud= 'aws'
        )
    )
    print('ir-multihop 인덱스 생성 완료!')
else:
    print('ir-multihop 인덱스 이미 존재!')

['ir-meta']
ir-multihop 인덱스 생성 완료!


In [ ]:
# DataFrame -> [Document, ...] 변환
from langchain_core.documents import Document # Langchain 문서 객체
from tqdm.auto import tqdm
from pprint import pprint
documents = []

for idx, row in tqdm(document_df.iterrows()):    # df를 행 단위로 순회
    doc_id = row['doc_id']
    content = row['content']
    doc = Document(
        page_content= content,
        metadata = {
            'doc_id': doc_id
        }
    )
    documents.append(doc)

pprint(documents)

0it [00:00, ?it/s]

[Document(metadata={'doc_id': 'D1'}, page_content='아이폰(iPhone) 스마트폰 시리즈는 애플(Apple Inc.)에 의해 설계 및 마케팅되었습니다.'),
 Document(metadata={'doc_id': 'D2'}, page_content='애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.'),
 Document(metadata={'doc_id': 'D3'}, page_content='쿠퍼티노(Cupertino)는 미국 캘리포니아주에 위치한 도시로, 애플의 본거지로 알려져 있습니다.'),
 Document(metadata={'doc_id': 'D4'}, page_content='팀 쿡(Tim Cook)은 2011년 스티브 잡스의 뒤를 이어 애플(Apple Inc.)의 CEO가 되었습니다.'),
 Document(metadata={'doc_id': 'D5'}, page_content='스티브 잡스(Steve Jobs)는 애플(Apple Inc.)의 공동 창업자였으며 2011년까지 CEO를 역임했습니다.'),
 Document(metadata={'doc_id': 'D6'}, page_content='갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.'),
 Document(metadata={'doc_id': 'D7'}, page_content='삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.'),
 Document(metadata={'doc_id': 'D8'}, page_content='이재용은 현재 삼성전자의 회장직을 맡고 있습니다.'),
 Document(metadata={'doc_id': 'D9'}, page_content='서울은 대한민국의 수도이자 최대 도시로, 한강이 흐르고 있습니다.'),
 Document(metadata={'doc_id': 'D10'}, page_content='삼성전자는 1938년 이병철에 의

In [6]:
# 원본 인덱스와 압축 인덱스 PineconeVectorStore 연결
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model='text-embedding-3-small') # 1536차원 임베딩 모델
# 벡터스토어 연결
vector_store = PineconeVectorStore(
    index_name = 'ir-multihop', # 연결할 index명
    embedding = embeddings # 사용할 임베딩 함수 (연결할 인덱스 차원과 임베딩 차원이 같아야함)
)

In [7]:
# Documents를 Pinecone에 업서트
vector_store.add_documents(documents)

['40e85a6b-ddb5-48fa-a229-4b8a9b25c6d5',
 'f740a8ab-4d84-4507-87a5-f3696b2dc09c',
 '6df623f7-6a6c-4b01-a012-6351e8795c5c',
 '398d5148-efb7-444e-b3ba-95ed72d55f79',
 '688e35f2-02a1-4e3d-9bd8-f4af073d861b',
 '1d57e672-496f-43c9-afa1-9f57d9a891e8',
 'f83e22f5-3f0c-477f-9069-ca5e8ce8bd99',
 '380af421-7ae6-4201-b6d9-ee667ee2149b',
 '7098938e-1bf4-497a-91e0-911283baa71a',
 '027b7f24-3e79-4bb9-b311-d6316abeb1fd',
 'c751a880-b631-4ff9-a456-3a1b47c7a244',
 'e0480714-99c0-47fb-a8b5-f0378d96227c',
 'c514e15f-2b2a-4629-9066-52b8fa82a9ae',
 '0e5ed123-3051-4c82-acde-32bb3e97910c',
 '0907f49e-f25e-4590-94d4-14bc86d6d08f',
 '5ddb5e47-479a-4d25-8bc8-944325c5b447',
 'd5ba5793-2318-48ec-a699-3c81adc56d37',
 '9ded8568-802b-4e79-ab37-f856ad5684e5',
 '6616a9bf-f1f0-4464-853e-f42c9549e14b',
 '7ba07047-81c1-46c5-aa60-4ba3f426ca5e',
 'd8fa7aa4-7144-470f-8b0f-13582f8c89e5',
 '49ff4ef1-d1d9-49fa-9819-23b92c6c7c92',
 '6d4e4920-fc12-4768-811d-f8b77cfc93b5',
 '87c7208a-1fae-4914-bc2d-a8d1a4f13f23',
 '10cc4da1-4152-

In [8]:
vector_store.similarity_search('손흥민', k=3)

[Document(id='edff4a5a-02c8-4bd7-bbea-0a3b6a98086f', metadata={'doc_id': 'D29'}, page_content='손흥민은 2021-2022 시즌 프리미어리그에서 아시아 선수 최초로 득점왕(골든 부트)을 수상했습니다.'),
 Document(id='7c276f17-52d3-438b-afe1-f37d43f848de', metadata={'doc_id': 'D26'}, page_content='손흥민은 잉글랜드 프리미어리그(EPL)의 토트넘 홋스퍼 FC에서 활약하는 대한민국 축구 선수입니다.'),
 Document(id='eca5596a-19d8-424a-b17a-f382b9ad407c', metadata={'doc_id': 'D30'}, page_content='해리 케인은 토트넘에서 손흥민과 환상적인 호흡을 보여준 잉글랜드 출신 스트라이커입니다.')]

### Multihop RAG 구현

In [9]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', temperature = 0)
prompt = ChatPromptTemplate.from_template('''
당신은 복잡한 질문에 답하기 위해 정보를 단계적으로 검색하는 AI에이젼트입니다.
사용자의 질문과 현재까지 수집된 정보를 바탕으로,
1. 아직 답을 찾지 못했다면:
  다음에 검색해야 할 가장 구체적이고 필요한 검색어(Query)를 출력하세요
2. 충분한 정보를 찾았다면:
  'ANSWER: '뒤에 최종 정답을 적어서 출력하세요.

### 사용자의 원래질문 ###
{original_question}

### 현재까지 수집된 정보 Context ###
{context}

### 출력지시사항 ###
불필요한 설명없이, '검색어' 또는 'ANSWER: 정답' 형식으로만 출력하세요.

1.추가검색이 필요한 경우, 검색어는 "BTS가 소속된 기획사"인 경우
("검색어" 출력하지 말것)
출력: BTS가 소속된 기획사

2.정답 추론이 가능한 경우
출력: ANSWER: 하이브
''')

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

print(chain.invoke({
    'original_question': 'BTS가 소속된 기획사를 설립한 사람이 누구인가요?',
    'context': ''
}))


ANSWER: 방시혁


### 멀티홉 질의응답 함수 구현

In [ ]:
# 멀티홉 검색을 통해 단계적으로 정답을 도출하는 함수
def multihop_search(question, max_hop=3):
    context = ''
    retrieved_doc_ids = set()

    for i in range(max_hop):
        print(f"현재 hop 단계는? {i+1}")
        response = chain.invoke({'original_question': question, 'context': context})
        # 정답 도출 여부 확인 : ANSWER로 시작하면 최종 정답
        if response.startswith('ANSWER:'):
            final_answer = response.replace('ANSWER','').strip()
            print(f"정답은? {final_answer}")
            return final_answer, retrieved_doc_ids
        # 검색 단계 : 에이전트 응답을 다음 검색어로 활용
        query = response
        docs = vector_store.similarity_search(query, k=3)
        for doc in docs:
            retrieved_doc_ids.add(doc.metadata['doc_id'])

        content = '\n'.join([doc.page_content for doc in docs])
        print(f"검색어 : {query}")
        print(f"검색결과 : {content}")

        if context == '':
            context = content
        else:
            context += '\n\n' + content

    print(f"{max_hop} 내에 정답을 찾지 못햇습니다.")
    return '검색 실패', retrieved_doc_ids

In [13]:
answer, retrieved_doc_ids = multihop_search('삼성전자 계열사')

print(answer)
print(retrieved_doc_ids)

현재 hop 단계는? 1
검색어 : 삼성전자 계열사 목록 2026
검색결과 : 삼성전자는 1938년 이병철에 의해 창립된 삼성그룹의 계열사입니다.
삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.
갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.
현재 hop 단계는? 2
검색어 : 삼성전자 계열사 목록 2025
검색결과 : 삼성전자는 1938년 이병철에 의해 창립된 삼성그룹의 계열사입니다.
삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.
갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.
현재 hop 단계는? 3
검색어 : 삼성전자 계열사 목록 2025
검색결과 : 삼성전자는 1938년 이병철에 의해 창립된 삼성그룹의 계열사입니다.
삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.
갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.
max_hop 내에 정답을 찾지 못햇습니다.
검색 실패
{'D7', 'D10', 'D6'}


In [18]:
answer, retrieved_doc_ids = multihop_search('영화 기생충이 상을 받은 영화제는?')

print(answer)
print(retrieved_doc_ids)

현재 hop 단계는? 1

검색어 : 영화 기생충이 수상한 영화제 목록
검색결과 : 봉준호 감독은 영화 '기생충'으로 칸 영화제에서 최고상인 황금종려상을 수상했습니다.
'기생충'은 제92회 아카데미 시상식(오스카)에서 작품상, 감독상 등 4관왕을 달성했습니다.
영화 '기생충(Parasite)'은 2019년 개봉한 봉준호 감독의 블랙 코미디 스릴러 영화입니다.
현재 hop 단계는? 2
봉준호 감독은 영화 '기생충'으로 칸 영화제에서 최고상인 황금종려상을 수상했습니다.
'기생충'은 제92회 아카데미 시상식(오스카)에서 작품상, 감독상 등 4관왕을 달성했습니다.
영화 '기생충(Parasite)'은 2019년 개봉한 봉준호 감독의 블랙 코미디 스릴러 영화입니다.
정답은? : 칸 영화제
: 칸 영화제
{'D21', 'D22', 'D24'}


### 핵심 결과
- 멀티홉 검색은 **단일 검색으로는 해결하기 어려운 복합 질의**에서 효과적으로 동작했다.
- 질문을 단계적으로 분해하고, 각 단계에서 필요한 정보를 순차적으로 수집함으로써  
  **추론 기반 질의에 대한 Recall을 안정적으로 확보**할 수 있었다.

---

### 왜 멀티홉이 필요한가
- 단일 Dense Retrieval은 **한 문서 안에 모든 단서가 존재한다는 가정**에 의존한다.
- 실제 질의는  
  - *“A와 관련된 B는 무엇인가?”*  
  - *“A를 만든 회사의 본사는 어디인가?”*  
  와 같이 **여러 문서에 정보가 분산**된 경우가 많다.
- 멀티홉 검색은 이 문제를  
  **검색 → 추론 → 추가 검색**의 반복 구조로 해결한다.

---

### 실험을 통해 확인된 장점
- 단계별 검색으로 **정답 문서 회수율(Recall) 향상**
- 검색 과정에서 실제로 참조된 문서를 추적 가능
- LLM이 “지금 무엇을 더 찾아야 하는지”를 스스로 판단하여  
  **질의 전개(Query Decomposition)**가 자연스럽게 수행됨

---

### 한계점
- 홉(hop) 수 증가에 따라 **지연 시간과 비용 증가**
- 중간 검색이 잘못되면 이후 단계도 함께 실패할 가능성 존재
- Precision보다는 **Recall 중심 평가에 더 적합**

### 실무 적용 결론
단일 검색으로 해결 가능 → 일반 Retrieval  
복합 추론이 필요한 질문 → Multi-hop Retrieval


- 멀티홉 검색은 **지식 탐색형 QA, 리서치, 에이전트 기반 RAG**에 특히 적합
- 실제 서비스에서는  
  **Self-Query / Metadata → Dense → Multi-hop → ReRank**  
  형태로 결합하는 것이 가장 현실적인 전략이다.

---

### 한 줄 결론 (교안 / 발표용)
> **멀티홉 검색은 분산된 정보를 단계적으로 연결하여, 복합 질의에 대한 추론 가능성을 확장하는 검색 전략이다.**